# RadHarmony — Usage Examples

| § | Feature |
|---|---|
| 1 | Harmonizers — standardized DataFrames from raw CSVs |
| 2 | Preprocessors — per-image and batch |
| 3 | Datasets — train / val split + sample inspection |
| 4 | Dataset options — output flags, masks, full, k-fold, no-cache, verify images |
| 5 | Multi-dataset — `ConcatDataset` |
| 6 | Dataset registry — `resolve_dataset`, `list_datasets`, `@register_dataset` |

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import SimpleITK as sitk
from PIL import Image

import monai as mn
from torch.utils.data import ConcatDataset

from radharmony.dataset import CTRATEDataset, MIMICCXRJPGDataset, SIIMACRPTXDataset
from radharmony.harmonizer import (
    CTRATEHarmonizer,
    MIMICCXRJPGHarmonizer,
    SIIMACRPTXHarmonizer,
)
from radharmony.preprocessor import CXRPreprocessor, CTPreprocessor
from radharmony.registry import register_dataset, resolve_dataset, list_datasets

# ── Image directories ─────────────────────────────────────────────────────────
CTRATE_DIR = "/path/to/CT-RATE/dataset/train_fixed/"
MIMIC_DIR = "/path/to/MIMIC_CXR/physionet.org/files/mimic-cxr-jpg/2.0.0/files/"
SIIM_DIR = "/path/to/SIIM_ACR_Pneumothorax/dicom-images-train/"
MASK_DIR = "/path/to/SIIM_ACR_Pneumothorax/masks/"
MIMIC_PP_DIR = "/path/to/MIMIC_CXR_radharmony_preprocessed/"

# ── CSV / metadata paths ──────────────────────────────────────────────────────
CTRATE_CSV = (
    "/path/to/CT-RATE/dataset/tables/train_predicted_labels.csv"
)
CTRATE_META_CSV = (
    "/path/to/CT-RATE/dataset/tables/train_metadata.csv"
)

MIMIC_ROOT = (
    "/path/to/MIMIC_CXR/physionet.org/files/mimic-cxr-jpg/2.0.0"
)
MIMIC_META_CSV = f"{MIMIC_ROOT}/mimic-cxr-2.0.0-metadata.csv"
MIMIC_LABEL_CSV = f"{MIMIC_ROOT}/mimic-cxr-2.0.0-chexpert.csv"

SIIM_RLE_CSV = "/path/to/SIIM_ACR_Pneumothorax/train-rle.csv"

## 1. Harmonizers

Each `*Harmonizer` class reads one or more raw dataset CSVs and returns a **standardised `pd.DataFrame`** — the shared contract used by all Dataset classes and custom pipelines. Calling `.harmonize()` is sufficient for most use cases; image and mask preprocessing are optional side-effects of the same call.

**Output columns** (always present):

| Column | Type | Description |
|---|---|---|
| `patient_id` | `str` | Patient identifier |
| `study_id` | `str` | Study / series identifier |
| `image_path` | `str` | Relative path from dataset root |

Optional columns appended when available: `view_position`, `mask_path`, `bbox`, then label columns in snake_case.

**`harmonize()` parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `preprocess_output_dir` | `str` | `None` | If set, runs the preprocessor over all images and rewrites `image_path` to the output `.nii` files |
| `preprocessor` | `RadiologicPreprocessor` | `None` | Required when `preprocess_output_dir` is set |
| `base_image_dir` | `str` | `None` | Prepended to relative image paths during preprocessing |
| `num_workers` | `int` | `1` | Parallel workers for image preprocessing |
| `sample_n` | `int` | `None` | Process only a random subset of this size |
| `mask_output_dir` | `str` | `None` | Decode RLE masks to PNGs and rewrite `mask_path` |
| `mask_num_cores` | `int` | `1` | Parallel threads for mask decoding |

### 1.1 MIMIC-CXR-JPG

`MIMICCXRJPGHarmonizer` joins the metadata CSV (`mimic-cxr-2.0.0-metadata.csv`) with the
CheXpert label CSV and builds structured `patient_id`, `study_id`, and `image_path` columns.


In [ ]:
df_mimic = MIMICCXRJPGHarmonizer(
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
).harmonize()

print(df_mimic.shape)
df_mimic.head()

### 1.2 CT-RATE

`CTRATEHarmonizer` derives `patient_id`, `study_id`, and `image_path` entirely from the `VolumeName` column in the labels CSV, then optionally merges the metadata CSV to add `view_position` (`ImageOrientationPatient`).

**Constructor parameters:**

| Parameter | Type | Description |
|---|---|---|
| `csv_path` | `str` | Path to `train_predicted_labels.csv` (contains `VolumeName` + 18 predicted label columns) |
| `view_position_csv_path` | `str` | Path to `train_metadata.csv` — omit to skip view position |

**Output DataFrame:**
- Shape: `(N_volumes, 3 + 1 + 18)` — `patient_id`, `study_id`, `image_path`, `view_position`, 18 label columns
- `image_path` format: `{patient}/{study}/{VolumeName}` (relative, prepend `base_image_dir`)
- `patient_id` = first two `_`-separated tokens of `VolumeName`; `study_id` = first three tokens
- 18 label columns (snake_case): `medical_material`, `arterial_wall_calcification`, `cardiomegaly`, `pericardial_effusion`, `coronary_artery_wall_calcification`, `hiatal_hernia`, `lymphadenopathy`, `emphysema`, `atelectasis`, `lung_nodule`, `lung_opacity`, `pulmonary_fibrotic_sequela`, `pleural_effusion`, `mosaic_attenuation_pattern`, `peribronchial_thickening`, `consolidation`, `bronchiectasis`, `interlobular_septal_thickening`

In [ ]:
df_ctrate = CTRATEHarmonizer(
    csv_path=CTRATE_CSV,
    view_position_csv_path=CTRATE_META_CSV,
).harmonize()

print(df_ctrate.shape)
df_ctrate.head()

### 1.3 SIIM-ACR Pneumothorax — decode RLE masks

`SIIMACRPTXHarmonizer` globs the DICOM tree to build `image_path`, derives a binary `pneumothorax` label from the `EncodedPixels` column (`-1` → 0, anything else → 1), and keeps the raw RLE string in `mask_path`.

Passing `mask_output_dir` to `.harmonize()` decodes those RLE strings into PNG files **once** and rewrites `mask_path` to the saved paths — subsequent calls skip already-existing PNGs.

**Constructor parameters:**

| Parameter | Type | Description |
|---|---|---|
| `csv_path` | `str` | Path to `train-rle.csv` (columns: `ImageId`, ` EncodedPixels`) |
| `dicom_dir` | `str` | Root directory of the DICOM tree; all `*.dcm` files are discovered recursively |

**`harmonize()` mask-related parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `mask_output_dir` | `str` | `None` | Directory to write per-image PNG masks; skipped if `None` |
| `mask_num_cores` | `int` | `1` | Parallel threads for RLE decoding |

**Output DataFrame columns:** `patient_id`, `study_id`, `image_path`, `view_position` (`"Frontal"`), `mask_path` (PNG path after decoding), `pneumothorax` (0/1)

> **Note:** Multiple annotation rows per image (multiple annotators) are unioned into a single binary mask before saving.

In [ ]:
df = SIIMACRPTXHarmonizer(
    csv_path=SIIM_RLE_CSV,
    dicom_dir=SIIM_DIR,
).harmonize(mask_output_dir=MASK_DIR, mask_num_cores=4)

# Sample 4 positive and 2 negative cases
pos = df[df["pneumothorax"] == 1].drop_duplicates("image_path").head(4)
neg = df[df["pneumothorax"] == 0].drop_duplicates("image_path").head(2)
samples = pd.concat([pos, neg]).reset_index(drop=True)

fig, axes = plt.subplots(len(samples), 3, figsize=(12, 4 * len(samples)))
for i, row in samples.iterrows():
    img = sitk.GetArrayFromImage(sitk.ReadImage(SIIM_DIR + row["image_path"]))
    if img.ndim == 3:
        img = img[0]
    mask = np.array(Image.open(row["mask_path"]))
    label = "positive" if row["pneumothorax"] else "negative"

    axes[i, 0].imshow(img, cmap="gray")
    axes[i, 0].set_title(f"Image ({label})")
    axes[i, 1].imshow(mask, cmap="gray")
    axes[i, 1].set_title("Mask")
    axes[i, 2].imshow(img, cmap="gray")
    axes[i, 2].imshow(mask, alpha=0.4, cmap="Reds")
    axes[i, 2].set_title("Overlay")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()

## 2. Preprocessors

`CXRPreprocessor` and `CTPreprocessor` standardise raw images to a fixed spatial size and intensity range before training. Both are callable (single image) and support batch processing via `run_batch()` / `harmonize(preprocess_output_dir=...)`.

| Preprocessor | Modality | Input formats | Pipeline steps | Output |
|---|---|---|---|---|
| `CXRPreprocessor` | 2-D X-ray | DICOM, PNG, JPEG | Modality LUT → VOI LUT (DICOM only) → grayscale float → rescale `[-1, 1]` → isotropic resize | `SimpleITK.Image` (2-D, float32) |
| `CTPreprocessor` | 3-D CT | NIfTI, NRRD, MHA, MHD | grayscale float → HU window → axial foreground crop → isotropic resize → `DICOMOrient` | `SimpleITK.Image` (3-D, float32) |

**Common parameters (both preprocessors):**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `target_long_side` | `int` | `112` | Target pixel/voxel count for the longest spatial dimension |
| `interpolation` | `sitk interpolator` | `sitkLinear` | SimpleITK resampling interpolator |

**`CTPreprocessor` additional parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `window_min` | `float` | `-1000` | Lower HU clip value |
| `window_max` | `float` | `1000` | Upper HU clip value |
| `orientation` | `str` | `"LIP"` | DICOM orientation code applied after resizing |

**Single-image** — call the preprocessor directly: `preprocessor(path)` → `SimpleITK.Image`.  
**Batch** — pass `preprocess_output_dir` + `preprocessor` to `.harmonize()`; images are written as `.nii` files and `image_path` in the returned DataFrame is updated.

### 2.1 Single-image

Call a preprocessor instance directly with a file path. Both 2-D and 3-D images are supported.

```python
cxr_out = CXRPreprocessor(target_long_side=224)(path)   # SimpleITK.Image (2-D)
ct_out  = CTPreprocessor(target_long_side=112)(path)    # SimpleITK.Image (3-D)
```

**Expected output shapes** (with `target_long_side=224` / `112`):
- CXR: longest side = 224 px, shorter side proportionally scaled, intensity ∈ `[-1, 1]`
- CT: longest axial side = 112 vx, depth unchanged after crop, intensity ∈ `[-1, 1]`

> Intensity is normalised to `[-1, 1]` in both cases: `CXRPreprocessor` uses the image's actual min/max; `CTPreprocessor` clips first to `[window_min, window_max]` then maps to `[-1, 1]`.

In [ ]:
# ── Single-image: CXR (2-D JPEG) ─────────────────────────────────────────────
cxr_prep = CXRPreprocessor(target_long_side=224)

cxr_path = MIMIC_DIR + df_mimic["image_path"].iloc[29]
cxr_out = cxr_prep(cxr_path)

arr_cxr = sitk.GetArrayFromImage(cxr_out)
print(f"CXR  size={cxr_out.GetSize()}  val=[{arr_cxr.min():.3f}, {arr_cxr.max():.3f}]")

# ── Single-image: CT (3-D NIfTI) ─────────────────────────────────────────────
ct_prep = CTPreprocessor(target_long_side=112, window_min=-1000, window_max=1000)

ct_path = CTRATE_DIR + df_ctrate["image_path"].iloc[0]
ct_raw = sitk.ReadImage(ct_path)
ct_out = ct_prep(ct_path)

arr_ct_raw = sitk.GetArrayFromImage(sitk.Cast(ct_raw, sitk.sitkFloat32)).mean(axis=0)
arr_ct_pp = sitk.GetArrayFromImage(ct_out).mean(axis=1)
print(
    f"CT   size={ct_out.GetSize()}  val=[{arr_ct_pp.min():.3f}, {arr_ct_pp.max():.3f}]"
)

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

axes[0, 0].imshow(sitk.GetArrayFromImage(sitk.ReadImage(cxr_path)), cmap="gray")
axes[0, 0].set_title("CXR — raw")
axes[0, 1].imshow(arr_cxr, cmap="gray")
axes[0, 1].set_title("CXR — preprocessed")
axes[1, 0].imshow(arr_ct_raw, cmap="gray")
axes[1, 0].set_title("CT — raw (mean projection)")
axes[1, 1].imshow(arr_ct_pp, cmap="gray")
axes[1, 1].set_title("CT — preprocessed (mean projection)")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()

### 2.2 Batch preprocessing

Pass `preprocess_output_dir` and `preprocessor` to `harmonize()` to preprocess the entire dataset in parallel. The harmonizer writes one `.nii` file per image and replaces `image_path` with the absolute output paths.

**Key parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `preprocess_output_dir` | `str` | — | Output directory for `.nii` files |
| `preprocessor` | `RadiologicPreprocessor` | — | `CXRPreprocessor` or `CTPreprocessor` instance |
| `base_image_dir` | `str` | `None` | Root prepended to relative `image_path` values when reading inputs |
| `num_workers` | `int` | `1` | Parallel worker processes |
| `sample_n` | `int` | `None` | Randomly sample this many images (useful for testing) |

**Output:** the returned DataFrame is identical to the standard harmonized DataFrame except `image_path` now contains **absolute** paths to the preprocessed `.nii` files (e.g. `{preprocess_output_dir}/{dicom_id}.nii`).

> Files that already exist in `preprocess_output_dir` are **not** recomputed — re-running is safe and fast.

In [ ]:
# ── Batch preprocessing via harmonizer ───────────────────────────────────────
df_pp = MIMICCXRJPGHarmonizer(
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
).harmonize(
    preprocess_output_dir=MIMIC_PP_DIR,
    preprocessor=CXRPreprocessor(target_long_side=224),
    base_image_dir=MIMIC_DIR,
    num_workers=2,
    sample_n=10,
)

print(df_pp.shape)
print("image_path sample:", df_pp["image_path"].iloc[0])

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (_, row) in zip(axes.flat, df_pp.iterrows()):
    arr = sitk.GetArrayFromImage(sitk.ReadImage(row["image_path"]))
    ax.imshow(arr, cmap="gray")
    ax.set_title(row["view_position"], fontsize=8)
    ax.axis("off")
plt.suptitle("Batch-preprocessed MIMIC-CXR samples")
plt.tight_layout()

In [ ]:
df_pp

## 3. Datasets

Dataset classes wrap a harmonizer and a MONAI transform pipeline to produce `torch`-ready samples. By default they use `monai.data.PersistentDataset` (disk-cached transforms); pass `cache_dir=None` for an in-memory `monai.data.Dataset`.

**Splitting is always patient-level** — no patient appears in both train and val.

**`get_datasets()` parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `n_splits` | `int` | `None` | Total folds; `1/n_splits` of patients go to val. Omit for a single full-dataset object |
| `num_cores` | `int` | `2` | Parallel workers for building data dicts |
| `random_state` | `int` | `56` | Seed for the patient-level shuffle |

**Returns:** a single `PersistentDataset` when `n_splits=None`, or `(train_dataset, val_dataset)` otherwise.

**Sample dict keys** depend on the output flags set at construction time (see §4.1). With `output_cls=True` the default keys are `img` and `cls`.

### 3.1 CT-RATE

3-D NIfTI chest CT volumes with 18 binary pathology labels predicted by a radiologist-validated model.

**Constructor parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `base_image_dir` | `str` | — | Root of the NIfTI volume tree (e.g. `…/dataset/train_fixed/`) |
| `csv_path` | `str` | auto-inferred | Path to `train_predicted_labels.csv` |
| `view_position_csv_path` | `str` | auto-inferred | Path to `train_metadata.csv` |
| `hu_window` | `tuple[float, float]` | `(-1000, 1000)` | HU clipping range applied before intensity normalisation |
| `cache_dir` | `str` | `"./cache"` | MONAI PersistentDataset cache root; `None` = no cache |

**Sample shape** (`output_cls=True`):
- `img`: `torch.bfloat16`, shape `(1, D, H, W)` — depth varies per volume; H and W ≤ `img_size` (default 112), padded to a multiple of `patch_size` (14)
- `cls`: `torch.bfloat16`, shape `(18,)` — one entry per `LABEL_COLS`

> CSV and metadata paths are auto-inferred from `<parent_of_base_image_dir>/tables/` if not provided.

In [ ]:
ctrate_ds_obj = CTRATEDataset(
    base_image_dir=CTRATE_DIR,
    csv_path=CTRATE_CSV,
    view_position_csv_path=CTRATE_META_CSV,
    cache_dir="./cache/ctrate",
    output_cls=True,
)
train_ds_ctrate, val_ds_ctrate = ctrate_ds_obj.get_datasets(n_splits=10, num_cores=2)

print(f"CT-RATE  — train: {len(train_ds_ctrate):,}  val: {len(val_ds_ctrate):,}")
print(f"Labels ({len(CTRATEDataset.LABEL_COLS)}): {CTRATEDataset.LABEL_COLS}")

### 3.2 MIMIC-CXR-JPG

2-D JPEG chest X-rays with 14 CheXpert pathology labels.


In [ ]:
mimic_ds_obj = MIMICCXRJPGDataset(
    base_image_dir=MIMIC_DIR,
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    cache_dir="./cache/mimic",
    output_cls=True,
)
train_ds_mimic, val_ds_mimic = mimic_ds_obj.get_datasets(n_splits=10, num_cores=2)

print(f"MIMIC-CXR — train: {len(train_ds_mimic):,}  val: {len(val_ds_mimic):,}")
print(f"Labels ({len(MIMICCXRJPGDataset.LABEL_COLS)}): {MIMICCXRJPGDataset.LABEL_COLS}")

### 3.3 Custom label subset

`LABEL_COLS` can be overridden after construction to restrict the `cls` tensor to a subset of labels — without rebuilding the dataset or invalidating the cache. The `img` tensor and all other keys are unaffected.

```python
ds.LABEL_COLS = ["atelectasis", "cardiomegaly"]  # cls shape becomes (2,)
```

> The cache is keyed on the input data dict, not on `LABEL_COLS`, so switching label subsets on a cached dataset is instant.

In [ ]:
mimic_ds = MIMICCXRJPGDataset(
    base_image_dir=MIMIC_DIR,
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    cache_dir="./cache/mimic",
    output_cls=True,
)

In [ ]:
# Override LABEL_COLS after construction — no cache invalidation
mimic_ds.LABEL_COLS = ["atelectasis", "cardiomegaly"]

mimic_subset_ds = mimic_ds.get_datasets(num_cores=2)
sample = mimic_subset_ds[0]

print(f"Dataset size : {len(mimic_subset_ds):,}")
print(f"Labels ({len(mimic_ds.LABEL_COLS)}) : {mimic_ds.LABEL_COLS}")
print(f"cls shape    : {tuple(sample['cls'].shape)}")

### 3.4 SIIM-ACR Pneumothorax

2-D DICOM chest X-rays with a single binary label (`pneumothorax`) and optional RLE-decoded segmentation masks.

**Constructor parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `base_image_dir` | `str` | — | Root of the DICOM tree (e.g. `…/dicom-images-train/`); also used for mask decoding |
| `csv_path` | `str` | auto-inferred | Path to `train-rle.csv` |
| `mask_output_dir` | `str` | `None` | If set, decodes RLE masks to PNGs here and enables `output_mask` |
| `mask_num_cores` | `int` | `1` | Parallel threads for mask decoding |
| `cache_dir` | `str` | `"./cache"` | MONAI PersistentDataset cache root; `None` = no cache |

**Sample shape** (`output_cls=True`):
- `img`: `torch.bfloat16`, shape `(1, H, W)` — longest side = 224, padded to a multiple of 14
- `cls`: `torch.bfloat16`, shape `(1,)` — single `pneumothorax` label

> VOI LUT windowing is applied automatically to DICOM inputs at load time.

In [ ]:
siim_ds_obj = SIIMACRPTXDataset(
    base_image_dir=SIIM_DIR,
    csv_path=SIIM_RLE_CSV,
    cache_dir="./cache/siim",
    output_cls=True,
)
train_ds_siim, val_ds_siim = siim_ds_obj.get_datasets(n_splits=10, num_cores=4)

print(f"SIIM-ACR  — train: {len(train_ds_siim):,}  val: {len(val_ds_siim):,}")
print(f"Labels ({len(SIIMACRPTXDataset.LABEL_COLS)}): {SIIMACRPTXDataset.LABEL_COLS}")

### 3.5 Inspect samples

Each sample is a `dict` returned by the MONAI transform pipeline. The keys present depend on which `output_*` flags were enabled at construction (see §4.1).

**With `output_cls=True`:**

| Dataset | `img` shape | `cls` shape | dtype |
|---|---|---|---|
| CT-RATE | `(1, D, 112, 112)` — D varies | `(18,)` | `bfloat16` |
| MIMIC-CXR | `(1, H, W)` — proportional resize | `(14,)` | `bfloat16` |
| SIIM-ACR | `(1, 224, 224)` | `(1,)` | `bfloat16` |

All pixel values are normalised to approximately `[-1, 1]`. Images are padded with fill value `-1` (image) or `0` (mask).

`visualize_samples` is a **generator** that yields `Figure` objects and adapts the layout to whichever outputs are active:

| Parameter | Default | Description |
|---|---|---|
| `n` | `4` | Number of random samples (ignored when `per_label=True` or `indices` given) |
| `indices` | `None` | Explicit list of dataset indices to display |
| `per_label` | `False` | One subplot grid figure with a positive example per label |
| `random_state` | `None` | Seed for reproducible random sampling |
| `projection_axis` | `0` | Axis used for mean-projection of 3-D volumes: `0`=axial, `1`=coronal, `2`=sagittal |

| Output key present | What is shown |
|---|---|
| `img` (always) | Grayscale image. 3-D volumes are collapsed via mean projection along `projection_axis`. |
| `cls` | Positive label names in the subplot title. |
| `mask` | Second subplot: image + red mask overlay. |
| `bbox` | Red rectangle drawn on the image subplot. |
| `report` | Report file path as a figure footnote. |

In [ ]:
for name, ds, label_cols in [
    ("CT-RATE", train_ds_ctrate, CTRATEDataset.LABEL_COLS),
    ("MIMIC-CXR", train_ds_mimic, MIMICCXRJPGDataset.LABEL_COLS),
    ("SIIM-ACR", train_ds_siim, SIIMACRPTXDataset.LABEL_COLS),
]:
    s = ds[0]
    active = [col for col, v in zip(label_cols, s["cls"].tolist()) if v]
    print(
        f"{name:10s}  img={tuple(s['img'].shape)}  cls={tuple(s['cls'].shape)}  positive={active or ['(none)']}"
    )

In [ ]:
from IPython.display import display

# per_label=True yields a single grid figure — one subplot per label class.
# CT-RATE is 3-D: projection_axis=0 (axial), 1 (coronal), 2 (sagittal).
# 2-D datasets ignore projection_axis.

for ds_obj, train_ds, title, proj_axis in [
    (ctrate_ds_obj, train_ds_ctrate, "CT-RATE", 0),
    (mimic_ds_obj, train_ds_mimic, "MIMIC-CXR", 0),
    (siim_ds_obj, train_ds_siim, "SIIM-ACR", 0),
]:
    print(f"───────────────────────────── {title} ─────────────────────────────")
    (fig,) = ds_obj.visualize_samples(
        train_ds, per_label=True, random_state=42, projection_axis=proj_axis
    )
    display(fig)
    plt.close(fig)

## 4. Dataset options

### 4.1 Output control flags

Boolean flags set at construction time control which keys appear in each sample dict. Only `"img"` is always present.

| Flag | Default | Sample key | Tensor shape | Notes |
|---|---|---|---|---|
| `output_cls` | `False` | `"cls"` | `(N_labels,)` bfloat16 | Multi-label float vector from `LABEL_COLS` |
| `output_mask` | `False` | `"mask"` | same as `"img"` | Requires `mask_path` in the harmonized DataFrame |
| `output_report` | `False` | `"report"` | string path | Requires `report_path` column |
| `output_bbox` | `False` | `"bbox"` | bfloat16 tensor | Requires `bbox` column |

When all flags are `False` the dataset returns `{"img": tensor}` only — useful for self-supervised or inference pipelines.

In [ ]:
# Image-only — drop the label vector
img_only_ds = SIIMACRPTXDataset(
    base_image_dir=SIIM_DIR,
    csv_path=SIIM_RLE_CSV,
    cache_dir=None,
    output_cls=False,
).get_datasets()

sample_img = img_only_ds[0]
print("output_cls=False →", list(sample_img.keys()))  # ['img']

### 4.2 Segmentation masks (`output_mask=True`)

Setting `output_mask=True` adds a `"mask"` tensor to every sample. The mask is loaded from the path stored in `mask_path`, resized to match `"img"` using nearest-neighbour interpolation, and padded with `0`.

For SIIM-ACR, pass `mask_output_dir` to `SIIMACRPTXDataset(...)` — this triggers RLE decoding on first construction and caches the PNGs for all subsequent runs.

**Sample dict with `output_cls=True, output_mask=True`:**

| Key | Shape | dtype | Description |
|---|---|---|---|
| `"img"` | `(1, H, W)` | `bfloat16` | Normalised X-ray |
| `"cls"` | `(1,)` | `bfloat16` | Binary `pneumothorax` label |
| `"mask"` | `(1, H, W)` | `bfloat16` | Binary segmentation mask (0 or 1) |

> Mask pixels are saved as 0/255 PNG but are normalised to 0/1 during loading.

In [ ]:
mask_ds = SIIMACRPTXDataset(
    base_image_dir=SIIM_DIR,
    csv_path=SIIM_RLE_CSV,
    mask_output_dir=MASK_DIR,
    cache_dir=None,
    output_cls=True,
    output_mask=True,
).get_datasets()

sample_mask = mask_ds[0]
print("Keys :", list(sample_mask.keys()))
print("img  :", sample_mask["img"].shape, sample_mask["img"].dtype)
print("mask :", sample_mask["mask"].shape, sample_mask["mask"].dtype)
print("cls  :", sample_mask["cls"].tolist())

### 4.3 Full dataset (no train/val split)

Omit `n_splits` from `get_datasets()` to receive a **single dataset** covering all patients. This is useful for inference, embedding extraction, or when you manage your own split externally.

```python
full_ds = SomeDataset(...).get_datasets()          # no n_splits → single dataset
train_ds, val_ds = SomeDataset(...).get_datasets(n_splits=10)  # with split
```

> With `n_splits=10`, approximately 10% of patients go to val and 90% to train.

In [ ]:
full_ds_mimic = MIMICCXRJPGDataset(
    base_image_dir=MIMIC_DIR,
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    cache_dir="./cache/mimic_full",
    output_cls=True,
).get_datasets()

print(f"MIMIC-CXR full: {len(full_ds_mimic):,}")

### 4.4 K-fold cross-validation

`get_folds(n_splits=N)` is a generator that yields one `(train_dataset, val_dataset)` tuple per fold. Patients are shuffled once and split into `N` equal buckets; each fold holds out one bucket as val.

**Parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `n_splits` | `int` | `5` | Number of folds |
| `num_cores` | `int` | `2` | Parallel workers for data dict construction |
| `random_state` | `int` | `56` | Seed for the patient-level shuffle |

Each fold uses a separate cache subdirectory (`fold_{i}/train` and `fold_{i}/val`) so all folds can be cached simultaneously without collision.

> The split is patient-level — studies from the same patient never appear in both train and val within a fold.

In [ ]:
for fold, (train_fold, val_fold) in enumerate(
    MIMICCXRJPGDataset(
        base_image_dir=MIMIC_DIR,
        csv_path=MIMIC_META_CSV,
        label_csv_path=MIMIC_LABEL_CSV,
        cache_dir="./cache/mimic_folds",
        output_cls=True,
    ).get_folds(n_splits=5, num_cores=4)
):
    print(f"Fold {fold}: train={len(train_fold):,}  val={len(val_fold):,}")

### 4.5 No cache (`cache_dir=None`)

Pass `cache_dir=None` to skip disk caching. The dataset uses `monai.data.Dataset` instead of `PersistentDataset`, meaning transforms are re-applied on every `__getitem__` call.

**When to use:**
- Quick prototyping or smoke-testing where cache warm-up latency is undesirable
- Storage-constrained environments
- When the transform pipeline is fast enough that caching provides no benefit

**Trade-off:** first-epoch latency is the same as subsequent epochs (no cold-start), but repeated access to the same sample is slower than with `PersistentDataset`.

> `type(train_nc)` → `monai.data.dataset.Dataset` (not `PersistentDataset`)

In [ ]:
train_nc, val_nc = MIMICCXRJPGDataset(
    base_image_dir=MIMIC_DIR,
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    cache_dir=None,
    output_cls=True,
).get_datasets(n_splits=10, num_cores=4)

print(type(train_nc))  # monai.data.dataset.Dataset

### 4.6 Pre-caching

`pre_cache(dataset)` iterates the entire dataset once to populate the `PersistentDataset` disk cache before training begins. Subsequent `__getitem__` calls load directly from disk instead of recomputing the transform pipeline.

**When to use:** on a machine with fast storage, warm the cache in a setup script or at the start of training so the first epoch is not bottlenecked by on-the-fly transforms.

**Parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `dataset` | `PersistentDataset` | — | Dataset to warm (returned by `get_datasets()` or `get_folds()`) |
| `num_workers` | `int` | `2` | Parallel DataLoader workers for concurrent cache writes |

> **Cache invalidation:** the cache key is a hash of the **input data dict**. Adding or removing keys (e.g. `mask_path`) changes the hash and forces a full re-cache. Changing only `LABEL_COLS` or the transform pipeline does **not** invalidate the cache.

In [ ]:
siim_ds = SIIMACRPTXDataset(
    base_image_dir=SIIM_DIR,
    csv_path=SIIM_RLE_CSV,
    cache_dir="./cache/siim_precache",
    output_cls=True,
)

train_ds, val_ds = siim_ds.get_datasets(n_splits=10, num_cores=4)

# Warm the cache before training — transforms are applied once and persisted to disk
siim_ds.pre_cache(train_ds, num_workers=4)
siim_ds.pre_cache(val_ds, num_workers=4)

print("Cache warm-up complete — subsequent DataLoader access will read from disk.")

### 4.7 Verify image files (`verify_images`)

Before building datasets, call `verify_images()` to check that every `image_path` in the harmonized table points to a file that actually exists on disk. With `drop_missing=True` (default), rows with missing files are removed so subsequent `get_datasets()` calls skip them.

This is available at both the **dataset** and **harmonizer** level.

In [ ]:
# ── Dataset level — uses base_image_dir automatically ──────────────────────
ds = MIMICCXRJPGDataset(
    base_image_dir=MIMIC_DIR,
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    cache_dir=None,
    output_cls=True,
)

# Check and drop rows with missing image files
missing = ds.verify_images()  # drop_missing=True by default
print(f"Missing files: {len(missing)}")

# Subsequent get_datasets() only uses rows with valid paths
train_ds, val_ds = ds.get_datasets(n_splits=5)
print(f"train: {len(train_ds):,}  val: {len(val_ds):,}")

# ── Inspect-only mode (don't modify the DataFrame) ─────────────────────────
# missing = ds.verify_images(drop_missing=False)

# ── Harmonizer level — pass base_image_dir explicitly ──────────────────────
# h = MIMICCXRJPGHarmonizer(csv_path=MIMIC_META_CSV, label_csv_path=MIMIC_LABEL_CSV)
# h.harmonize()
# missing = h.verify_images(base_image_dir=MIMIC_DIR)

## 5. Multi-dataset

`torch.utils.data.ConcatDataset` combines multiple dataset objects into a single iterable with a unified integer index space. This is the recommended way to train on data from more than one source simultaneously.

**Requirements and caveats:**

- All constituent datasets must produce sample dicts with the **same keys** (controlled by the `output_*` flags) and **compatible tensor shapes** — e.g. mixing 2-D and 3-D `img` tensors will break a standard `DataLoader` collate.
- If `output_cls=True`, the `cls` tensors must have the **same length** across datasets, or you must handle heterogeneous labels in a custom collate function.
- Each dataset manages its own cache independently; there is no shared cache for the combined dataset.

```python
combined = ConcatDataset([train_ds_a, train_ds_b])
len(combined)  # == len(train_ds_a) + len(train_ds_b)
```

In [ ]:
combined = ConcatDataset([train_ds_mimic, train_ds_siim])

print(f"MIMIC-CXR train : {len(train_ds_mimic):,}")
print(f"SIIM-ACR  train : {len(train_ds_siim):,}")
print(f"Combined        : {len(combined):,}")

## 6. Dataset registry

The registry maps string names to dataset classes. This decouples training configs from `import` statements — any dataset can be instantiated by name, including custom ones defined outside the package.

| Function | Description |
|---|---|
| `list_datasets()` | Returns a sorted list of all registered dataset names |
| `resolve_dataset(name)` | Returns the class registered under *name* |
| `@register_dataset(name)` | Decorator to register a new dataset class |

All three built-in datasets (`"mimic_cxr"`, `"ct_rate"`, `"siim_acr_ptx"`) are registered automatically when `radharmony.dataset` is imported.

### 6.1 Discover and instantiate built-in datasets

`list_datasets()` shows every registered name. `resolve_dataset(name)` returns the class — call it exactly as you would the class directly.

In [ ]:
print("Registered datasets:", list_datasets())

# resolve_dataset returns the class — identical to importing it directly
DatasetCls = resolve_dataset("mimic_cxr_jpg")
print("Class:", DatasetCls)
print("Label columns:", DatasetCls.LABEL_COLS)

### 6.2 Config-driven instantiation

The main benefit of the registry is that the dataset name can come from a config dict or YAML file, making training scripts fully data-driven without any hard-coded imports.

In [ ]:
# Simulate a config loaded from YAML / argparse / Hydra
cfg = {
    "dataset": "mimic_cxr_jpg",
    "base_image_dir": MIMIC_DIR,
    "csv_path": MIMIC_META_CSV,
    "label_csv_path": MIMIC_LABEL_CSV,
    "cache_dir": "./cache/mimic_registry",
    "output_cls": True,
}

DatasetCls = resolve_dataset(cfg.pop("dataset"))  # resolve name → class
train_ds, val_ds = DatasetCls(**cfg).get_datasets(n_splits=10, num_cores=4)

print(f"train: {len(train_ds):,}  val: {len(val_ds):,}")
print("Sample keys:", list(train_ds[0].keys()))

### 6.3 Registering a custom dataset

Decorate any `BaseRadiologicalDataset` subclass with `@register_dataset("name")`. The class registers itself at import time and then behaves identically to the built-in datasets — including appearing in `list_datasets()` and being retrievable via `resolve_dataset()`.

Set **`_HARMONIZER_CLS`** on the class to your harmonizer type so callers can use **`harmonizer_path=`** (a file from `BaseHarmonizer.save`). In `_get_harmonized_df`, call **`_try_resolve_preset_harmonized()`** first and return that DataFrame when not `None`; otherwise run your usual harmonize logic. If you post-filter rows (as in the PA-only example below), presets must already match that filter unless you re-apply filtering after the preset branch.

In [ ]:
from radharmony.dataset import BaseRadiologicalDataset, RadiologyTransform2D
from radharmony.harmonizer import MIMICCXRJPGHarmonizer


@register_dataset("mimic_cxr_jpg_pa_only")
class MIMICCXRJPGPADataset(BaseRadiologicalDataset):
    """MIMIC-CXR-JPG restricted to PA (postero-anterior) views only."""

    LABEL_COLS = MIMICCXRJPGDataset.LABEL_COLS
    _HARMONIZER_CLS = MIMICCXRJPGHarmonizer

    def __init__(
        self,
        base_image_dir,
        csv_path=None,
        label_csv_path=None,
        transform=None,
        cache_dir="./cache",
        output_cls=False,
        harmonized_df=None,
        harmonizer=None,
        harmonizer_path=None,
    ):
        output_keys = {"img", "cls"} if output_cls else {"img"}
        super().__init__(
            base_image_dir=base_image_dir,
            transform=transform
            or RadiologyTransform2D(output_keys=output_keys).get_transform(),
            cache_dir=cache_dir,
            output_cls=output_cls,
            harmonized_df=harmonized_df,
            harmonizer=harmonizer,
            harmonizer_path=harmonizer_path,
        )
        self._harmonizer = MIMICCXRJPGHarmonizer(
            csv_path=csv_path, label_csv_path=label_csv_path
        )

    def _get_harmonized_df(self):
        preset = self._try_resolve_preset_harmonized()
        if preset is not None:
            return preset
        df = self._harmonizer.harmonize()
        return df[df["view_position"] == "PA"].reset_index(drop=True)


# Confirm it is now part of the registry
print("Registered datasets:", list_datasets())

# Use it the same way as any built-in dataset
DatasetCls = resolve_dataset("mimic_cxr_jpg_pa_only")
pa_ds = DatasetCls(
    base_image_dir=MIMIC_DIR,
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    cache_dir="./cache/mimic_pa",
    output_cls=True,
).get_datasets()

print(f"PA-only dataset size: {len(pa_ds):,}")

## 7. Transform builder

`RadiologyTransform2D` / `RadiologyTransform3D` expose an API for constructing the full preprocessing + augmentation pipeline.  
The pipeline is always: **load → normalize → resize → pad → augmentations → to-tensor → select keys**.

All `with_*` methods accept a `prob` argument (default 0.5) and return `self`, so calls can be chained.  
Call `.get_transform()` at the end to produce a `monai.transforms.Compose` object.

### 7.1 Preprocessing only (no augmentation)

Pass `transform=` explicitly, or omit it — the dataset builds a default preprocessing-only transform internally.

In [ ]:
from radharmony.dataset import MIMICCXRJPGDataset, RadiologyTransform2D

# Preprocessing only — equivalent to the dataset's internal default
transform = RadiologyTransform2D(
    img_size=224, output_keys={"img", "cls"}
).get_transform()

ds = MIMICCXRJPGDataset(
    base_image_dir=MIMIC_DIR,
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    transform=transform,
    cache_dir="./cache/mimic_transforms",
    output_cls=True,  # need to match output_keys
)
train_ds, val_ds = ds.get_datasets(n_splits=10, num_cores=4)

sample = train_ds[0]
print("Keys  :", list(sample.keys()))
print("Image :", sample["img"].shape, sample["img"].dtype)
print("Labels:", sample["cls"].shape)

### 7.2 Separate train / val transforms

The dataset returns MONAI `PersistentDataset` objects whose `.transform` attribute can be replaced after splitting.

In [ ]:
from radharmony.dataset import MIMICCXRJPGDataset, RadiologyTransform2D

train_transform = (
    RadiologyTransform2D(img_size=224, output_keys={"img", "cls"})
    .with_flip(spatial_axis=1, prob=0.5)  # left-right flip
    .with_affine(
        translate_range=(10, 10), rotate_range=(0.17,), scale_range=(0.1, 0.1), prob=0.5
    )
    .with_intensity_jitter(shift=0.1, scale=0.1)
    .with_gaussian_noise(std=0.03)
    .get_transform()
)

val_transform = RadiologyTransform2D(
    img_size=224, output_keys={"img", "cls"}
).get_transform()  # no augmentation

ds = MIMICCXRJPGDataset(
    base_image_dir=MIMIC_DIR,
    csv_path=MIMIC_META_CSV,
    label_csv_path=MIMIC_LABEL_CSV,
    cache_dir="./cache/mimic_aug",
    output_cls=True,
)
train_ds, val_ds = ds.get_datasets(
    n_splits=10,
    num_cores=4,
    train_transform=train_transform,
    val_transform=val_transform,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N = 5  # images per row

fig, axes = plt.subplots(2, N, figsize=(3 * N, 7))

for col in range(N):
    for row, (ds, title) in enumerate(
        [(train_ds, "Train (augmented)"), (val_ds, "Val (no aug)")]
    ):
        sample = ds[col]
        # (1, H, W) bfloat16 in [-1, 1]  →  (H, W) float32 in [0, 1]
        img = sample["img"].float().squeeze(0).numpy()
        img = np.clip((img + 1.0) / 2.0, 0.0, 1.0)

        ax = axes[row, col]
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
        if col == 0:
            ax.set_title(title, fontsize=10, loc="left", pad=4)

plt.suptitle("Train vs Val — same indices, different transforms", y=1.01)
plt.tight_layout()
plt.show()

### 7.3 2D with mask (SIIM-ACR-PTX)

When `output_mask=True` are set, include `"mask"` in `output_keys` so the transform pipeline resizes them alongside the image.

In [ ]:
from radharmony.dataset import SIIMACRPTXDataset, RadiologyTransform2D

transform = (
    RadiologyTransform2D(
        img_size=512,
        output_keys={"img", "cls", "mask"},
    )
    .with_flip(spatial_axis=1, prob=0.5)
    .with_affine(translate_range=(15, 15), scale_range=(0.1, 0.1), prob=0.5)
    # spacing: grid control point distance (px); magnitude must stay well below spacing
    .with_elastic_deformation(spacing=100, magnitude_range=(5, 15), prob=0.2)
    .get_transform()
)

ds = SIIMACRPTXDataset(
    base_image_dir=SIIM_DIR,
    csv_path=SIIM_RLE_CSV,
    transform=transform,
    cache_dir="./cache/siim_mask",
    mask_output_dir=MASK_DIR,
    output_cls=True,
    output_mask=True,
    output_bbox=False,
)
train_ds, val_ds = ds.get_datasets(n_splits=10, num_cores=4)

sample = train_ds[0]
print("Keys :", list(sample.keys()))
print("Image:", sample["img"].shape)
print("Mask :", sample["mask"].shape)

In [ ]:
for fig in ds.visualize_samples(val_ds, n=6):
    display(fig)
    plt.close(fig)

### 7.4 3D CT with HU windowing (CT-RATE)

`hu_window=(min_HU, max_HU)` clips the Hounsfield range before percentile normalisation.  
`with_elastic_deformation` on the 3D builder uses `sigma_range` (controls deformation field smoothness) instead of `spacing`.

In [ ]:
from radharmony.dataset import CTRATEDataset, RadiologyTransform3D

transform = (
    RadiologyTransform3D(
        img_size=112,
        output_keys={"img", "cls"},
        hu_window=(-1000, 400),  # lung window
    )
    .with_flip(spatial_axis=0, prob=0.5)
    .with_affine(translate_range=(5, 5, 5), rotate_range=(0.09, 0.09, 0.09), prob=0.5)
    .with_gaussian_smooth(sigma_range=(0.5, 1.0), prob=0.2)
    .with_elastic_deformation(
        sigma_range=(5.0, 7.0),
        magnitude_range=(50, 150),
        prob=0.2,
    )
    .get_transform()
)

ds = CTRATEDataset(
    base_image_dir=CTRATE_DIR,
    csv_path=CTRATE_CSV,
    view_position_csv_path=CTRATE_META_CSV,
    transform=transform,
    cache_dir="./cache/ct_rate_aug",
    output_cls=True,
)
train_ds, val_ds = ds.get_datasets(n_splits=10, num_cores=4)

sample = train_ds[0]
print("Keys  :", list(sample.keys()))
print("Volume:", sample["img"].shape, sample["img"].dtype)
print("Labels:", sample["cls"].shape)

### 7.5 Escape hatch — inserting a custom MONAI transform

Use `.add_transform()` to inject any MONAI transform not covered by the built-in `with_*` methods.

In [ ]:
import monai as mn
from radharmony.dataset import RadiologyTransform2D

transform = (
    RadiologyTransform2D(img_size=224, output_keys={"img"})
    .with_flip(spatial_axis=1)
    .add_transform(  # any MONAI transform
        mn.transforms.RandAffineD(
            keys=["img"],
            prob=0.3,
            translate_range=(10, 10),
            mode="bilinear",
        )
    )
    .get_transform()
)

print(type(transform))
print(f"{len(transform.transforms)} steps in the pipeline")